In [0]:
# from datetime import datetime, timedelta

# today = datetime.today()

# # Latest Friday (today if Friday, else previous Friday)
# latest_friday = today - timedelta(days=(today.weekday() - 4) % 7)

# # T-4 Friday (4 weeks = 28 days before latest Friday)
# t4_friday = latest_friday - timedelta(days=28)
# t3_friday = latest_friday - timedelta(days=21)

# START_END_DATE = t4_friday.strftime("%Y-%m-%d")
# END_END_DATE   = t3_friday.strftime("%Y-%m-%d")
# STEP_DAYS      = 7 

# print("END_END_DATE T-3 Friday:", END_END_DATE)
# print("START_END_DATE T-4 Friday:", START_END_DATE)


In [0]:
from datetime import datetime, timedelta

# ============================================================
# CONFIG
# ============================================================

START_END_DATE = "2025-08-08"
END_END_DATE   = "2026-03-06"
STEP_DAYS      = 7

DB_TEMP = "ccex_tmp"

# Source tables
T_MAU   = "ccex_tmp.okr_segment_tags_final_base" # new base table
T_NUF   = "ccex.ccex_nuf_weekly_temp"  # first_event_ts source (NEW/NUV)
T_TRON  = "ccex.ccex_tron_activity_props"
T_SPARK = "spark.spark_tron_activity_props"
T_AU_Temp = "ccex.ccex_au_temp"

# Temp tables (overwritten per end_date)
T_BASE          = f"{DB_TEMP}.li_mau_base_dedup_tmp"
T_FIRST_TS      = f"{DB_TEMP}.li_first_ts_tmp"
T_ACTIVITY      = f"{DB_TEMP}.li_activity_tmp"
T_EXPORT_ACT    = f"{DB_TEMP}.li_export_activity_tmp"
T_KPI_EDITOR    = f"{DB_TEMP}.li_kpi_editor_tmp"
T_KPI_EXPORT    = f"{DB_TEMP}.li_kpi_export_tmp"
T_KPI_RETURNS   = f"{DB_TEMP}.li_kpi_returns_tmp"
T_USER_LEVEL    = f"{DB_TEMP}.li_user_level_tmp"
T_NUV_USER_LEVEL = f"{DB_TEMP}.li_nuv_user_level_tmp"
T_ENGAGED_MAU   = f"{DB_TEMP}.li_engaged_mau_tmp" # engaged mau

# Output table 

T_OUT_METRICS = f"{DB_TEMP}.core_leading_indicator_metrics_mau_v2"

#### events tables
ccex_tron_activity_props_DB = "ccex"
ccex_tron_activity_props_tbl = "ccex_tron_activity_props"
ccex_event_activity_props_tbl = "ccex_event_activity_b"

spark_tron_activity_props_DB = "spark"
spark_tron_activity_props_tbl = "spark_tron_activity_props"


# ============================================================
# NON CONSENT CONFIG — Add new tags here as needed!
# ============================================================
NON_CONSENT_TAGS = [
    'non-consent',
    'standalone'
]

# Non consent table
T_PRECONSENT = "ccex.ccex_tron_preconsent_deduped_mau"

# Build SQL IN list dynamically
NON_CONSENT_TAGS_SQL = ", ".join(
    [f'"{tag}"' for tag in NON_CONSENT_TAGS]
)

#-------------------------------------------------------------------------------

# -------------------------
# Segment logic
# -------------------------

# 1) Core filter: MUST use columns available in MAU table.
#    Replace this string with your actual "Core" logic.
CORE_FILTER_SQL = """
source_name IN ('CCEX','1.N')
AND lower(coalesce(platform_category,"")) != 'dapp-windows'
AND lower(coalesce(offer_category,"")) NOT LIKE '%edu%'
"""

CC_FILTER_SQL = """m.cloud_type_value = 'Creative Cloud Subs'
        AND b.market_segment_category = 'Adobe Members Entitled with Express Premium'
        AND lower(coalesce(b.contract_type,"")) <> 'edu enterprise k12'
        AND lower(coalesce(b.offer_type,"")) <> 'acrobat sa'
        """

third_party_filter_sql = """ lower(channel_detail) like '%third party%' """

# 2) okr_subsegment values for Acrobat photos / non-photos
#    Fill these lists with the exact okr_subsegment values used in your MAU table.
ACROBAT_PHOTOS_SUBSEGMENTS = ["Acrobat - Photos"]
ACROBAT_NONPHOTOS_SUBSEGMENTS = ["Acrobat - Non Photos"]
cc_entitled = ['CC Entitled']
third_party = [
    '3P Integrations',
    '3P Telecom Companies: Airtel India'
]
edu = ['Edu']

# 3) Segment registry: add new segments by adding a dict entry
#    Each entry should define:
#      - name: segment label to appear in output
#      - where: SQL predicate evaluated on user-level rows (base table flags)

SEGMENTS = [
    {"name": "OVERALL",            "where": "1=1"},
    {"name": "Core",               "where": "core_flag = 1"},
    {"name": "Acrobat photos",     "where": "acrobat_photos_flag = 1"},
    {"name": "Acrobat non-photos", "where": "acrobat_nonphotos_flag = 1"},
    {"name": "edu",                "where": "edu_flag=1"},
    {"name": "cc entitled",        "where": "cc_entitled_flag=1"},
    {"name": "3p",                 "where": "third_party_flag=1"}
]

# ============================================================
# Helpers 
# ============================================================

def exec_sql(sql: str):
    spark.sql(sql)

def daterange(start_date: str, end_date: str, step_days: int):
    s = datetime.strptime(start_date, "%Y-%m-%d").date()
    e = datetime.strptime(end_date, "%Y-%m-%d").date()
    d = s
    while d <= e:
        yield d.strftime("%Y-%m-%d")
        d += timedelta(days=step_days)

# def ensure_output_tables():
#     exec_sql(f"""CREATE 
#              or replace table
#     --table IF NOT EXISTS
#      {T_OUT_METRICS} (
#         platform_cut          STRING,
#         segment               STRING,
#         MAU                   BIGINT,
#         pnuv_users            BIGINT,
#         fta                   DOUBLE,
#         d1_editor_pct         DOUBLE,
#         d1_export_pct         DOUBLE,
#         export_pct      DOUBLE,
#         d2_7_pct              DOUBLE,
#         m1_rr_pct             DOUBLE,
#         rmau_pct              DOUBLE
#     )
#     PARTITIONED BY (end_date DATE)
#     """)


# def ensure_output_tables():
#     exec_sql(f"""
#     CREATE --or replace table
#     TABLE IF NOT EXISTS 
#     {T_OUT_METRICS} (
#         platform_cut          STRING,
#         new_or_return     STRING,
#         market_area      STRING,
#         auth_status       STRING,

#         segment               STRING,

#         MAU                   BIGINT,
#         pnuv_users            BIGINT,
#         d1_fta                   DOUBLE,
#         d1_editor_pct         DOUBLE,
#         d1_export_pct         DOUBLE,
#         export_pct            DOUBLE,
#         d2_7_pct              DOUBLE,
#         w1_rr_pct    DOUBLE,
#         m1_rr_pct             DOUBLE,
#         rmau_pct              DOUBLE
#     )
#     PARTITIONED BY (end_date DATE)
#     """)

def ensure_output_tables():
    exec_sql(f"""
    CREATE --or replace table
    TABLE IF NOT EXISTS 
    {T_OUT_METRICS} (

        -- ======================
        -- Dimensions
        -- ======================
        platform_cut      STRING,
        new_or_return     STRING,
        market_area       STRING,
        auth_status       STRING,
        segment           STRING,

        -- ======================
        -- Base volumes
        -- ======================
        MAU               BIGINT,
        pnuv_users        BIGINT,
    


        -- ✅ NEW KPI: NUV (NUF last-28d distinct users)
        -- Note: populated only for (segment/new_or_return/market_area/auth_status) = OVERALL
        -- and platform_cut in OVERALL, each platform
        nuv_users         BIGINT,
        total_visitors BIGINT,

        -- ======================
        -- FTA
        -- ======================
        d1_fta_users      BIGINT,
        d1_fta_pct        DOUBLE,


        fta_users   BIGINT,
        fta_pct     DOUBLE,
      
    
        -- ======================
        -- Editor engagement
        -- ======================
        d1_editor_users   BIGINT,
        d1_editor_pct     DOUBLE,

        editor_users    BIGINT,
        editor_pct   DOUBLE,

        -- ======================
        -- Export
        -- ======================
        d1_export_users   BIGINT,
        d1_export_pct     DOUBLE,

        export_users      BIGINT,
        export_pct        DOUBLE,

        -- ======================
        -- Short-term return
        -- ======================
        d2_7_users        BIGINT,
        d2_7_pct          DOUBLE,

        -- ======================
        -- Week-1 retention (WAU-based)
        -- ======================
        w1_rr_users       BIGINT,
        w1_rr_pct         DOUBLE,

        -- ======================
        -- Month-1 retention
        -- ======================
        m1_rr_users       BIGINT,
        m1_rr_pct         DOUBLE,

        -- ======================
        -- rMAU retention
        -- ======================
        rmau_users        BIGINT,
        rmau_pct          DOUBLE
    )
    PARTITIONED BY (end_date DATE)
    """)

ensure_output_tables()

In [0]:
# DB_TEMP="ccex_tmp"

# =========================================================
# 1) STATIC TEMP VIEW(S)
# =========================================================
def create_market_segment_view():
    market_seg_df = (
        spark.table("spark.market_segment_categorization")
        .selectExpr(
            "offer_category_key",
            "sku_code_primary",
            "offer_category",
            "offer_subcategory",
            "contract_type",
            "sku_market_segment_primary",
            "cloud_type",
            "sku_cc_segment_primary as cc_segment",
            "explode(market_segment_category) as market_segment_category",
            "offer_type",
            "dme_acct_segment"
        )
        .distinct()
    )
    market_seg_df.createOrReplaceTempView("ms_categorization")

create_market_segment_view()

# =========================================================
# 2) Acrobat /Airtel TEMP VIEW(S)
# =========================================================

airtel_events=""" 
      --airtel

      'access-from-deeplink',

      'access-app-complete',
      'open-editor','authentication:loginSucceeded','project:editorDisplayed',
      'view-express-home','homePageViewed',
      'view-quickaction-upload-page','quickAction:uploadPageViewed','pageload-complete',
      'initialize-app-launch','open-published-link','view-community-wall',
      'select-cover-page-service-start'


"""
airtel = '''
-- Airtel flag as 0/1 int 
  MAX(
  CASE
    WHEN e.user_properties['custom.user.offer_id'] = 'DE3F04C37860C5E1DA0BD640AF1BF621'
      OR e.event_properties['custom.link.id'] = 'partnerpath'
    THEN 1 ELSE 0
  END
) AS airtel_offer_id

'''

acrobat_events = """   
      'start-import-media',
      'open-editor','view-quickaction-upload-page','select-quickaction-asset','view-community-wall',
      'pageload-complete','access-app-complete',
      'access-from-deeplink',
      'initialize-app-launch',
      'copy-link-url','select-template','generate-presentation-complete','invite-sent',
      'editor-asset-load-complete'
      
      """

acrobat_flags = '''/* ---------- Acrobat/Integration flags ---------- */
  MAX(CASE WHEN 
             e.event_name in ( 'open-editor','pageload-complete','access-app-complete')
             and  e.event_properties['custom.sdk.client_name'] = 'Adobe Acrobat Extension' THEN 1 ELSE 0 END) = 1
    AS is_acrobat_extension_user,

  MAX(CASE 
         WHEN 
             e.event_name = 'initialize-app-launch'
            AND e.user_properties['hz.source_platform_type'] = 'desktop-app'
            AND e.user_properties['custom.user.installation_source'] <> 'ccd'
           THEN 1 ELSE 0 END) = 1
    AS is_adobe_express_photos_user,

  MAX(CASE WHEN e.event_name IN ('access-app-complete','pageload-complete','open-editor','invite-sent','copy-link-url','select-template','generate-presentation-complete')
            AND e.event_properties['custom.sdk.workflow_intent'] = 'edit-generative-presentation'
           THEN 1 ELSE 0 END) = 1
    AS is_generative_reuse_user,

  MAX(CASE WHEN e.event_properties['custom.sdk.client_id'] = 'ReaderMobileAndroid4_0002'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%create%'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%recent%'
            AND lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) NOT LIKE '%na%'
           THEN 1 ELSE 0 END) = 1
    AS is_acrobat_mobile_edit_image_user,

  MAX(CASE WHEN (
            (
              e.event_properties['custom.sdk.client_id'] IN (
                'AdobeReader9','AdobeReader9RCM','AdobeReader9PPTRCM','AdobeReader9OutlookRCM',
                'AdobeAcrobat9','AdobeAcrobat9PPTRCM','AdobeAcrobat9RCM','AdobeAcrobat9OutlookRCM','dc-prod-virgoweb'
              )
              AND e.event_properties['custom.sdk.workflow_intent'] IN (
                'image-module-v2','text-to-image-module','text-to-image-module-v2','search-and-generate-workflow','image-module'
              )
            )
            OR
            (
              e.event_name = 'start-import-media'
              AND e.event_properties['custom.content.upload_method'] = 'transform-upload'
              AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%acrobat%'
            )
          ) THEN 1 ELSE 0 END) = 1
    AS is_enhanced_pdf_editing_user,

  MAX(CASE WHEN (
            (
              e.event_properties['custom.sdk.client_id'] = 'ReaderMobileAndroid4_0002'
              AND (
                lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%create%'
                OR lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%recent%'
                OR lower(coalesce(e.event_properties['custom.sdk.client_cta_location'],'na')) LIKE '%na%'
              )
            )
            OR
            (
              e.event_name IN ('access-app-complete','pageload-complete','open-editor','select-template')
              AND e.event_properties['custom.referrer.app'] IN ('acrobat-web-studio','acrobat-desktop-studio','acrobat-reader-studio')
            )
            OR
            (
              e.event_name IN ('access-app-complete','pageload-complete','open-editor')
              AND e.event_properties['custom.sdk.client_cta_location'] IN (
                'create','studio-home','suggested-tools','all-tools','create-menu',
                'prompt-bar-generate','prompt-bar-tools','create-menu-dropdown','create-tab-card','all-tools-create-section-card'
              )
              AND e.event_properties['custom.sdk.client_id'] IN ('AdobeAcrobat9','dc-prod-virgoweb')
            )
            OR
            (e.event_name = 'start-import-media' AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%studio%')
            OR
            (e.event_name = 'start-import-media' AND lower(coalesce(e.event_properties['custom.referrer.app_intent'],'')) LIKE '%studio%')
          ) THEN 1 ELSE 0 END) = 1
    AS is_acrobat_original_creation_user,

  MAX(CASE WHEN (
      (e.event_name = 'start-import-media'
        AND e.event_properties['custom.content.upload_method'] = 'transform-upload'
        AND lower(coalesce(e.event_properties['custom.referrer.app'],'')) LIKE '%acrobat%')
      OR
      (e.event_name IN ('open-editor','view-quickaction-upload-page','select-quickaction-asset','view-community-wall')
        AND e.event_properties['custom.sdk.client_name'] IN ('Acrobat','Adobe Acrobat Extension'))
      OR
      (e.event_name IN ('pageload-complete','access-app-complete')
        AND e.event_properties['custom.sdk.client_name'] IN ('Acrobat','Adobe Acrobat Extension')
        AND e.auth_flag = 'true')
      OR
      (e.event_name = 'access-from-deeplink'
        AND e.event_properties['custom.link.full'] IN (
          'https://adobesparkpost.app.link/83JRwB0j9Qb',
          'https://adobesparkpost.app.link/n7T79GOk9Qb',
          'https://adobesparkpost.app.link/mRjdnB4mDRb',
          'https://adobesparkpost.app.link/f54E0t01ZLb',
          'https://adobesparkpost.app.link/mBBoTGm2ZLb',
          'https://adobesparkpost.app.link/c8iaL3disSb',
          'https://adobesparkpost.app.link/oJcM4qKHsSb'
        )
        AND e.auth_flag = 'true')
      OR
      (e.event_name = 'initialize-app-launch'
        AND e.user_properties['hz.source_platform_type'] = 'desktop-app'
        AND e.user_properties['custom.user.installation_source'] <> 'ccd')
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor','copy-link-url','select-template','generate-presentation-complete')
        AND e.event_properties['custom.sdk.workflow_intent'] = 'edit-generative-presentation')
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor')
        AND e.event_properties['custom.referrer.app'] IN ('acrobat-web-studio','acrobat-desktop-studio'))
      OR
      (e.event_name IN ('access-app-complete','pageload-complete','open-editor')
        AND e.event_properties['custom.sdk.client_cta_location'] IN (
          'create','studio-home','suggested-tools','all-tools','create-menu',
          'prompt-bar-generate','prompt-bar-tools','create-menu-dropdown','create-tab-card'
        )
        AND e.event_properties['custom.sdk.client_id'] IN ('AdobeAcrobat9','dc-prod-virgoweb'))
      OR
      (e.event_name = 'start-import-media'
        AND lower(coalesce(e.event_properties['custom.referrer.app_intent'],'')) LIKE '%studio%')
    ) THEN 1 ELSE 0 END) = 1 AS is_acrobat_integration_user,

------non consent  Acrobat
MAX(CASE WHEN e.event_name in ('editor-asset-load-complete') THEN 1 ELSE 0 END)=1 AS is_acrobat_non_consent
'''

def build_event_activity_view(end_date: str,period=28):
    event_activity_sql = f"""
    WITH auth AS (
      SELECT
        event_user_guid,
        max(e.user_properties['hz.source_platform_type']) as source_platform_type,
        max(e.user_properties['custom.user.installation_source']) as installation_source,
        {acrobat_flags},
        {airtel}
       
      FROM {ccex_tron_activity_props_DB}.{ccex_event_activity_props_tbl} e
      WHERE event_date BETWEEN DATE_SUB('{end_date}', {period}-1) AND '{end_date}'
        AND auth_flag = true
        AND event_name IN ({acrobat_events} ,{airtel_events}
                         )
      GROUP BY ALL
    ),
    unauth AS (
      SELECT
        event_visitor_guid as event_user_guid,
        max(e.user_properties['hz.source_platform_type']) as source_platform_type,
        max(e.user_properties['custom.user.installation_source']) as installation_source,
        {acrobat_flags},
        {airtel}
  
      FROM {ccex_tron_activity_props_DB}.{ccex_event_activity_props_tbl} e
      WHERE event_date BETWEEN DATE_SUB('{end_date}', {period}-1) AND '{end_date}'
        AND auth_flag = false
        AND event_name IN ({acrobat_events},{airtel_events}
        )
      GROUP BY ALL
    )
    SELECT * FROM auth
    UNION ALL
    SELECT * FROM unauth
    """
    spark.sql(event_activity_sql).createOrReplaceTempView("event_activity_on_date_vw")

In [0]:
# ============================================================
# 1) Base: dedupe exploded MAU table
# ============================================================

def build_base_dedup(end_date: str):
    photos_in = ", ".join([f"'{x}'" for x in ACROBAT_PHOTOS_SUBSEGMENTS]) or "''"
    nonphotos_in = ", ".join([f"'{x}'" for x in ACROBAT_NONPHOTOS_SUBSEGMENTS]) or "''"
    edu_in = ", ".join([f"'{x}'" for x in edu]) or "''"
    cc_entitled_in = ", ".join([f"'{x}'" for x in cc_entitled]) or "''"
    third_party_in = ", ".join([f"'{x}'" for x in third_party]) or "''"

    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_BASE} AS
    SELECT
      date_date AS end_date,
      user_guid,
      platform_category AS platform_cut,
      new_or_return,
      market_area,

      -- pNUV membership
      MAX(CASE WHEN new_or_return IN ('NEW','NUV')
          THEN 1 ELSE 0 END) AS pnuv_flag,

      -- T4 return flags
      MAX(CASE WHEN TRIM(COALESCE(nuv_active_t4_friday,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_t4_return_flag,
      MAX(CASE WHEN TRIM(COALESCE(mau_active_t4_friday,'')) <> ''
          THEN 1 ELSE 0 END) AS mau_t4_return_flag,

      -- Photos specific flags
      MAX(CASE WHEN TRIM(COALESCE(nuv_users_segment,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_photos_flag,
      MAX(CASE WHEN TRIM(COALESCE(nuv_active_t4_friday_segment,'')) <> ''
          THEN 1 ELSE 0 END) AS pnuv_photos_t4_return_flag,
      MAX(new_or_return_photos) AS new_or_return_photos,

      -- Segment flags
      MAX(CASE WHEN ({CORE_FILTER_SQL})
          THEN 1 ELSE 0 END) AS core_flag,

      -- segment_tag replaces okr_subsegment
      MAX(CASE WHEN segment_tag IN ({photos_in})
          THEN 1 ELSE 0 END) AS acrobat_photos_flag,
      MAX(CASE WHEN segment_tag IN ({nonphotos_in})
          THEN 1 ELSE 0 END) AS acrobat_nonphotos_flag,

      -- segment_tag replaces okr_segment
      MAX(CASE WHEN segment_tag IN ({edu_in})
          THEN 1 ELSE 0 END) AS edu_flag,
      MAX(CASE WHEN segment_tag IN ({cc_entitled_in})
          THEN 1 ELSE 0 END) AS cc_entitled_flag,
      MAX(CASE WHEN segment_tag IN ({third_party_in})
          THEN 1 ELSE 0 END) AS third_party_flag

    FROM {T_MAU} m
    LEFT ANTI JOIN (
        SELECT * FROM {T_PRECONSENT}
        WHERE date_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    ) pc
      ON m.user_guid = pc.visitor_guid
      AND lower(m.segment_tag) IN ({NON_CONSENT_TAGS_SQL})

    WHERE m.date_date = '{end_date}'

    GROUP BY all
    """)


# ============================================================
# 2) Join first_event_ts + auth_flag for pNUV users
# ============================================================

def build_first_ts(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_FIRST_TS} AS
    WITH nuf AS (
      SELECT
        user_guid,
        MAX(auth_flag) AS auth_flag,
        MIN(event_ts)  AS first_event_ts
      FROM {T_NUF}
      WHERE date_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      GROUP BY user_guid
    ),

    nuf_wau AS (
      SELECT
        distinct a.user_guid,
        1 pnuv_wau
      FROM {T_NUF} a inner join {T_AU_Temp} b
      on a.user_guid=b.user_guid
      WHERE a.date_date ='{end_date}'
    )
    SELECT
      b.*,
      CASE WHEN b.pnuv_flag = 1 THEN n.first_event_ts ELSE NULL END AS first_event_ts,
      CASE WHEN b.pnuv_flag = 1 THEN n.auth_flag      ELSE NULL END AS auth_flag,
      nvl(nw.pnuv_wau,0) pnuv_wau

    FROM {T_BASE} b
    LEFT JOIN nuf n ON b.user_guid = n.user_guid
    LEFT JOIN nuf_wau nw ON b.user_guid = nw.user_guid
    """)


# ============================================================
# 3) Activity union
# ============================================================

def build_activity(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_ACTIVITY} AS

    -- CCEX auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date,
      platform        AS ea_platform,
      auth_flag       AS ea_auth_flag,
      event_name,
      source_name     AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND source_name IN ('ESDK','CCEX')
      AND auth_flag = true
      AND (
            event_name IN ('access-app-complete','view-express-home','open-editor',
                           'view-quickaction-upload-page','view-quickaction-editor')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR (event_name = 'initialize-app-launch' AND event_properties['source.client_id'] = 'harmony-win-service')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.first_visit_page_url']) LIKE '%/template%'
             AND event_date>='2026-01-31')
      )

    UNION ALL

    -- CCEX auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date,
      platform           AS ea_platform,
      auth_flag          AS ea_auth_flag,
      event_name,
      source_name        AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND source_name IN ('ESDK','CCEX')
      AND auth_flag = false
      AND (
            event_name IN ('access-app-complete','view-express-home','open-editor',
                           'view-quickaction-upload-page','view-quickaction-editor')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR (event_name = 'initialize-app-launch' AND event_properties['source.client_id'] = 'harmony-win-service')
         OR (event_name = 'pageload-complete' AND LOWER(event_properties['event.first_visit_page_url']) LIKE '%/template%'
             AND event_date>='2026-01-31')
      )

    UNION ALL

    -- 1.N auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date,
      platform        AS ea_platform,
      auth_flag       AS ea_auth_flag,
      event_name,
      source_name     AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND auth_flag = true
      AND event_name IN ('experiment:standard:unauthenticated:assigned','authenticated:unknownEventName')
      AND (
            (LOWER(platform)='web' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR LOWER(platform)='mobile'
      )

    UNION ALL

    -- 1.N auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date,
      platform           AS ea_platform,
      auth_flag          AS ea_auth_flag,
      event_name,
      source_name        AS ea_source_name,
      event_properties['event.url'] AS event_url
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND date_add('{end_date}',7)
      AND auth_flag = false
      AND event_name IN ('experiment:standard:unauthenticated:assigned','authenticated:unknownEventName')
      AND (
            (LOWER(platform)='web' AND LOWER(event_properties['event.url']) LIKE '%express.adobe.com%')
         OR LOWER(platform)='mobile'
      )
    """)


# ============================================================
# 4) Export activity union
# ============================================================

def build_export_activity(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_EXPORT_ACT} AS

    -- CCEX auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = true
      AND (
        (
          event_date < '2023-09-06'
          AND event_name IN (
              'export-project-complete',
              'select-composer-publish-now-option',
              'select-post-composer-schedule',
              'export-quickaction-complete',
              'create-template',
              'publish-embed-project-complete'
          )
        )
        OR
        (
          event_date >= '2023-09-06'
          AND event_name IN ('export-project-complete','publish-embed-project-complete')
        )
      )

    UNION ALL

    -- CCEX auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date
    FROM {T_TRON}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = false
      AND event_name IN (
        'export-project-complete-unauth',
        'export-project-complete',
        'publish-embed-project-complete',
        'select-composer-publish-now-option',
        'select-post-composer-schedule',
        'export-quickaction-complete'
      )

    UNION ALL

    -- 1.N auth=true
    SELECT DISTINCT
      event_user_guid AS ea_user_guid,
      event_dts       AS ea_event_ts,
      event_date      AS ea_event_date
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = true
      AND event_name IN ('libraries:templateCreated','project:exportCompleted','contentCal:postScheduled','contentCal:postPublished')

    UNION ALL

    -- 1.N auth=false
    SELECT DISTINCT
      event_visitor_guid AS ea_user_guid,
      event_dts          AS ea_event_ts,
      event_date         AS ea_event_date
    FROM {T_SPARK}
    WHERE event_date BETWEEN date_sub('{end_date}',27) AND '{end_date}'
      AND auth_flag = false
      AND event_name IN ('libraries:templateCreated','project:exportCompleted','contentCal:postScheduled','contentCal:postPublished')
    """)


# ============================================================
# 5) KPI: D1 editor open (pNUV only)
# ============================================================

def build_kpi_editor(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_EDITOR} AS
    WITH b2 AS (
      SELECT
        user_guid,
        pnuv_flag,
        COALESCE(
          try_to_timestamp(first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AS first_ts
      FROM {T_FIRST_TS}
    ),
    a2 AS (
      SELECT
        ea_user_guid,
        COALESCE(
          try_to_timestamp(ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AS ea_ts,
        event_name
      FROM {T_ACTIVITY}
    )
    SELECT
      b2.user_guid,

      MAX(
        CASE
          WHEN b2.pnuv_flag = 1
           AND b2.first_ts IS NOT NULL
           AND a2.ea_ts IS NOT NULL
           AND a2.ea_ts BETWEEN b2.first_ts AND b2.first_ts + INTERVAL 1 DAY
           AND a2.event_name IN ('open-editor','view-quickaction-upload-page','view-quickaction-editor')
          THEN 1 ELSE 0
        END
      ) AS d1_editor_open,

      MAX(
        CASE
          WHEN a2.ea_ts IS NOT NULL
           AND to_date(a2.ea_ts) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
           AND a2.event_name IN ('open-editor','view-quickaction-upload-page','view-quickaction-editor')
          THEN 1 ELSE 0
        END
      ) AS total_editor_open

    FROM b2
    LEFT JOIN a2 ON b2.user_guid = a2.ea_user_guid
    GROUP BY b2.user_guid;
    """)


# ============================================================
# 6) KPI: D1 export count -> flags (pNUV only)
# ============================================================

def build_kpi_export(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_EXPORT} AS
    SELECT
      b.user_guid,
      max(CASE
        WHEN b.pnuv_flag = 1 AND
        COALESCE(
          try_to_timestamp(e.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(e.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) BETWEEN COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AND COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) + INTERVAL 1 DAY
        THEN 1 ELSE 0 END
      ) AS d1_export_cnt,

      max(CASE
        WHEN to_date(e.ea_event_ts) BETWEEN date_sub(to_date('{end_date}'), 27) AND to_date('{end_date}')
        THEN 1 ELSE 0 END
      ) AS total_export_cnt

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_EXPORT_ACT} e ON b.user_guid = e.ea_user_guid
    GROUP BY b.user_guid
    """)


# ============================================================
# 7) KPI: Return flags (D2-7, W1)
# ============================================================

def build_kpi_returns(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_KPI_RETURNS} AS
    SELECT
      b.user_guid,

      MAX(CASE
        WHEN b.pnuv_flag = 1
        AND COALESCE(
          try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) >= COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) + INTERVAL 1 DAY
        AND a.ea_event_ts <= COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) + INTERVAL 7 DAY
        THEN 1 ELSE 0 END
      ) AS d2_7_return_flag,

      MAX(CASE
        WHEN a.ea_auth_flag=true
        AND to_date(a.ea_event_ts) BETWEEN date_sub('{end_date}',27) AND to_date('{end_date}')
        THEN 1 ELSE 0 END
      ) AS auth_flag,

      MAX(CASE
        WHEN a.ea_auth_flag=true
        AND COALESCE(
          try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(a.ea_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) BETWEEN COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) AND COALESCE(
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH:mm:ss.SSS'),
          try_to_timestamp(b.first_event_ts, 'yyyy-MM-dd HH.mm.ss.SSS')
        ) + INTERVAL 1 DAY
        THEN 1 ELSE 0 END
      ) AS d1_auth_flag,

      MAX(CASE
        WHEN b.pnuv_flag = 1
        AND pnuv_wau=1
        AND to_date(a.ea_event_ts) BETWEEN date_add('{end_date}',1) AND date_add('{end_date}',7)
        THEN 1 ELSE 0 END
      ) AS w1_return_flag

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_ACTIVITY} a ON b.user_guid = a.ea_user_guid
    GROUP BY b.user_guid
    """)


# ============================================================
# Engaged MAU
# ============================================================

def build_engaged_mau(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_ENGAGED_MAU} AS

    -- Auth users
    SELECT DISTINCT event_user_guid AS user_guid
    FROM ccex.ccex_tron_activity_props
    WHERE event_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    AND auth_flag = true
    AND (
        lower(event_name) IN (
            'add-asset','add-page','add-text','adjust-animation','adjust-border',
            'adjust-chart-settings','adjust-edge','adjust-effect','adjust-grid',
            'adjust-opacity','adjust-shape-effect','adjust-text-outline-thickness',
            'adjust-video-speed','apply-adjustments','apply-animation',
            'apply-auto-caption-color','apply-auto-caption-color-outline',
            'apply-auto-caption-shape-color','apply-auto-caption-shape-color-shadow',
            'apply-auto-caption-style','apply-auto-enhance','apply-ai-style',
            'apply-blend-mode','apply-border-type','apply-bulk-create',
            'apply-color-palette','apply-crop','apply-crop-page',
            'apply-document-style','apply-duration-to-all-scenes','apply-effects',
            'apply-filter','apply-font-recommendation','apply-generated-image',
            'apply-gradient-color','apply-library-asset','apply-loop-type',
            'apply-page-transition','apply-resize-page-variation','apply-scene-duration',
            'apply-shadow-effect','apply-shape-effect','apply-styles-to-all-pages',
            'apply-template-style','apply-text-effect','apply-text-effect-fit',
            'apply-text-effect-font','apply-text-outline','apply-to-all-font-rec',
            'apply-transition','apply-transition-to-all-pages',
            'apply-transition-to-all-scenes','apply-webpage-custom-theme',
            'apply-webpage-theme','auto-increase-scene-duration','change-page-name',
            'collapse-brand-colors','complete-av-upload','complete-remove-background',
            'complete-voice-recording','copy-scene','create-library',
            'create-new-file-from-template','create-webpage-custom-theme','cut-scene',
            'delete-page','detach-background','disable-text-fill','disable-text-outline',
            'duplicate-page','duplicate-resize-page','duplicate-scene',
            'duplicate-webpage-custom-theme','edit-slide-outline','edit-text-entity',
            'edit-webpage-custom-theme','edit-within-text-editor','expand-brand-colors',
            'fit-video','flip-content','generate-copywriter-assistant-result',
            'generate-image','generate-image-fill','generate-presentation',
            'generate-recommendation','generate-video','insert-webpage-button',
            'insert-webpage-element','insert-webpage-gif','insert-webpage-glideshow',
            'insert-webpage-photo','insert-webpage-photo-grid','insert-webpage-text',
            'modify-object-time','modify-volume-audio','modify-volume-video',
            'move-caption','move-content','mute-all-video','mute-video',
            'open-animation-panel','open-color-panel','open-insert-object-ai-panel',
            'open-remove-object-ai-panel','open-resize-panel','open-video-panel',
            'paste-scene','reorder-asset','reorder-scene','remove-all-animation',
            'remove-duration','remove-hyperlink','replace-asset','replace-background',
            'replace-linked-asset','replace-text','reset-genfill-image','resize-content',
            'resize-page','rotate-content','save-template','search-all','search-inspire',
            'select-add-page-transition','select-add-scene','select-add-transition',
            'select-align','select-asset','select-auto-enhance','select-bleed-toggle',
            'select-color','select-color-applied','select-color-eyedropper',
            'select-color-eyedropper-sample','select-color-format','select-color-palette',
            'select-copywriter-assistant-rewrite','select-create-collage','select-crop',
            'select-cutout-done','select-document-style','select-erase',
            'select-fit-to-content','select-font','select-generate-outline',
            'select-generate-slide','select-generative-fill',
            'select-generative-fill-upload','select-hyperlink',
            'select-insert-webpage-element','select-margins-toggle','select-page-styles',
            'select-page-visibility','select-remove-background','select-replace-asset',
            'select-restore-background','select-restore-brush','select-restore-image',
            'select-restore-video','select-social-safezone','select-task-within-resize',
            'select-template','select-text-effects','select-text-to-image',
            'select-translate-page','select-webpage-themes','set-as-background',
            'set-background-color','set-new-hyperlink','set-text-font-fallback',
            'slip-edit-video','split-scene','start-av-upload',
            'start-content-canvas-render','start-convert-to-presentation',
            'translate-page-complete','trim-audio','trim-scene','unmute-all-video',
            'unmute-video','upload-content-complete','upload-font','view-genai-results'
        )
        OR (lower(event_name) = 'select-remove-background'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-crop'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-crop'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-insert-object-ai-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-remove-object-ai-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-erase'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'rotate-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-adjust-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-effects-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'adjust-opacity'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'flip-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-editor'
            AND lower(event_properties['custom.sdk.workflow_intent']) IN ('remove-background','crop-image')
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-crop-option'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'adjust-corner-radius'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'generate-image'
            AND lower(event_properties['custom.sdk.prompt_location']) != 'sdk-client-surface'
            AND lower(event_properties['custom.sdk.client_name']) LIKE '%acrobat%')
        OR (lower(event_name) = 'view-genai-results'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-effects'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-adjustments'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-cutout-done'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'scale-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'edit-text-entity'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-transform-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'start-text-editor'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'resize-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'generate-image-fill'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'add-asset'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'content-transform-complete'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'change-zoom'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_subcategory) = 'edit'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-desktop','acrobat-desktop-web'))
        OR (lower(event_name) = 'select-cover-page-service-start')
        OR (lower(event_name) = 'generate-presentation-complete')
        OR (lower(event_name) = 'view-image' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'export-project-complete' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'open-editor' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'select-template'
            AND lower(event_properties['custom.ui.location']) = 'explore'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat-desktop-studio','acrobat-web-studio'))
        OR (lower(event_properties['event.subcategory']) = 'edit'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-dekstop','acrobat-desktop-web'))
        OR (lower(event_name) = 'publish-embed-project-complete'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'export-project-complete'
            AND lower(event_properties['custom.sdk.client_name']) = 'acrobat')
        OR (lower(event_name) = 'apply-filter'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-auto-enhance'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'export-project-complete'
            AND lower(event_properties['custom.referrer.app']) LIKE '%acrobat%')
    )

    UNION

    -- Unauth users (same conditions)
    SELECT DISTINCT event_visitor_guid AS user_guid
    FROM ccex.ccex_tron_activity_props
    WHERE event_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    AND auth_flag = false
    AND (
        lower(event_name) IN (
            'add-asset','add-page','add-text','adjust-animation','adjust-border',
            'adjust-chart-settings','adjust-edge','adjust-effect','adjust-grid',
            'adjust-opacity','adjust-shape-effect','adjust-text-outline-thickness',
            'adjust-video-speed','apply-adjustments','apply-animation',
            'apply-auto-caption-color','apply-auto-caption-color-outline',
            'apply-auto-caption-shape-color','apply-auto-caption-shape-color-shadow',
            'apply-auto-caption-style','apply-auto-enhance','apply-ai-style',
            'apply-blend-mode','apply-border-type','apply-bulk-create',
            'apply-color-palette','apply-crop','apply-crop-page',
            'apply-document-style','apply-duration-to-all-scenes','apply-effects',
            'apply-filter','apply-font-recommendation','apply-generated-image',
            'apply-gradient-color','apply-library-asset','apply-loop-type',
            'apply-page-transition','apply-resize-page-variation','apply-scene-duration',
            'apply-shadow-effect','apply-shape-effect','apply-styles-to-all-pages',
            'apply-template-style','apply-text-effect','apply-text-effect-fit',
            'apply-text-effect-font','apply-text-outline','apply-to-all-font-rec',
            'apply-transition','apply-transition-to-all-pages',
            'apply-transition-to-all-scenes','apply-webpage-custom-theme',
            'apply-webpage-theme','auto-increase-scene-duration','change-page-name',
            'collapse-brand-colors','complete-av-upload','complete-remove-background',
            'complete-voice-recording','copy-scene','create-library',
            'create-new-file-from-template','create-webpage-custom-theme','cut-scene',
            'delete-page','detach-background','disable-text-fill','disable-text-outline',
            'duplicate-page','duplicate-resize-page','duplicate-scene',
            'duplicate-webpage-custom-theme','edit-slide-outline','edit-text-entity',
            'edit-webpage-custom-theme','edit-within-text-editor','expand-brand-colors',
            'fit-video','flip-content','generate-copywriter-assistant-result',
            'generate-image','generate-image-fill','generate-presentation',
            'generate-recommendation','generate-video','insert-webpage-button',
            'insert-webpage-element','insert-webpage-gif','insert-webpage-glideshow',
            'insert-webpage-photo','insert-webpage-photo-grid','insert-webpage-text',
            'modify-object-time','modify-volume-audio','modify-volume-video',
            'move-caption','move-content','mute-all-video','mute-video',
            'open-animation-panel','open-color-panel','open-insert-object-ai-panel',
            'open-remove-object-ai-panel','open-resize-panel','open-video-panel',
            'paste-scene','reorder-asset','reorder-scene','remove-all-animation',
            'remove-duration','remove-hyperlink','replace-asset','replace-background',
            'replace-linked-asset','replace-text','reset-genfill-image','resize-content',
            'resize-page','rotate-content','save-template','search-all','search-inspire',
            'select-add-page-transition','select-add-scene','select-add-transition',
            'select-align','select-asset','select-auto-enhance','select-bleed-toggle',
            'select-color','select-color-applied','select-color-eyedropper',
            'select-color-eyedropper-sample','select-color-format','select-color-palette',
            'select-copywriter-assistant-rewrite','select-create-collage','select-crop',
            'select-cutout-done','select-document-style','select-erase',
            'select-fit-to-content','select-font','select-generate-outline',
            'select-generate-slide','select-generative-fill',
            'select-generative-fill-upload','select-hyperlink',
            'select-insert-webpage-element','select-margins-toggle','select-page-styles',
            'select-page-visibility','select-remove-background','select-replace-asset',
            'select-restore-background','select-restore-brush','select-restore-image',
            'select-restore-video','select-social-safezone','select-task-within-resize',
            'select-template','select-text-effects','select-text-to-image',
            'select-translate-page','select-webpage-themes','set-as-background',
            'set-background-color','set-new-hyperlink','set-text-font-fallback',
            'slip-edit-video','split-scene','start-av-upload',
            'start-content-canvas-render','start-convert-to-presentation',
            'translate-page-complete','trim-audio','trim-scene','unmute-all-video',
            'unmute-video','upload-content-complete','upload-font','view-genai-results'
        )
        OR (lower(event_name) = 'select-remove-background'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-crop'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-crop'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-insert-object-ai-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-remove-object-ai-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-erase'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'rotate-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-adjust-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-effects-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'adjust-opacity'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'flip-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-editor'
            AND lower(event_properties['custom.sdk.workflow_intent']) IN ('remove-background','crop-image')
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-crop-option'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'adjust-corner-radius'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'generate-image'
            AND lower(event_properties['custom.sdk.prompt_location']) != 'sdk-client-surface'
            AND lower(event_properties['custom.sdk.client_name']) LIKE '%acrobat%')
        OR (lower(event_name) = 'view-genai-results'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-effects'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-adjustments'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'select-cutout-done'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'scale-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'edit-text-entity'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'open-transform-panel'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'start-text-editor'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'resize-content'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'generate-image-fill'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'add-asset'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'content-transform-complete'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'change-zoom'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_subcategory) = 'edit'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-desktop','acrobat-desktop-web'))
        OR (lower(event_name) = 'select-cover-page-service-start')
        OR (lower(event_name) = 'generate-presentation-complete')
        OR (lower(event_name) = 'view-image' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'export-project-complete' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'open-editor' AND lower(platform_category) = 'desktop-app')
        OR (lower(event_name) = 'select-template'
            AND lower(event_properties['custom.ui.location']) = 'explore'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat-desktop-studio','acrobat-web-studio'))
        OR (lower(event_properties['event.subcategory']) = 'edit'
            AND lower(event_properties['custom.referrer.app']) IN ('acrobat','acrobat-dekstop','acrobat-desktop-web'))
        OR (lower(event_name) = 'publish-embed-project-complete'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'export-project-complete'
            AND lower(event_properties['custom.sdk.client_name']) = 'acrobat')
        OR (lower(event_name) = 'apply-filter'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'apply-auto-enhance'
            AND lower(event_properties['custom.sdk.client_name']) IN ('acrobat','adobe acrobat extension'))
        OR (lower(event_name) = 'export-project-complete'
            AND lower(event_properties['custom.referrer.app']) LIKE '%acrobat%')
    )
    """)


# ============================================================
# 8) User-level join table (dedup base + KPI flags)
# ============================================================

def build_user_level(end_date: str):
    exec_sql(f"""
    CREATE OR REPLACE TABLE {T_USER_LEVEL} AS
    SELECT DISTINCT

      b.end_date,
      b.user_guid,
      b.platform_cut,
      b.new_or_return,

      CASE WHEN b.market_area NOT IN ('US', 'UK', 'IN', 'BRZ')
           THEN 'ROW' ELSE market_area
      END AS market_area,

      CASE
        WHEN COALESCE(rr.auth_flag,0) = 1 THEN 'authenticated'
        ELSE 'unauthenticated'
      END AS auth_status,

      rr.d1_auth_flag,
      rr.auth_flag AS mau_auth_flag,

      b.pnuv_flag,
      b.pnuv_wau,

      b.pnuv_t4_return_flag,
      b.mau_t4_return_flag,
      b.auth_flag AS nuv_auth_flag,

      --- Photos specific flags (Requirement 2)
      b.pnuv_photos_flag,
      b.pnuv_photos_t4_return_flag,
      b.new_or_return_photos,

      b.core_flag,
      b.acrobat_photos_flag,
      b.acrobat_nonphotos_flag,
      b.edu_flag,
      b.cc_entitled_flag,
      b.third_party_flag,

      COALESCE(ed.d1_editor_open,0) AS d1_editor_open,
      COALESCE(ed.total_editor_open,0) AS total_editor_open,

      CASE WHEN COALESCE(ex.d1_export_cnt,0) >= 1 THEN 1 ELSE 0 END AS d1_export_flag,
      CASE WHEN COALESCE(ex.total_export_cnt,0) >= 1 THEN 1 ELSE 0 END AS total_export_flag,

      COALESCE(rr.d2_7_return_flag,0) AS d2_7_return_flag,
      COALESCE(rr.w1_return_flag,0) AS w1_return_flag,

      --- NEW: Engaged MAU flag (Requirement 3)
      CASE WHEN eng.user_guid IS NOT NULL
           THEN 1 ELSE 0 END AS engaged_mau_flag

    FROM {T_FIRST_TS} b
    LEFT JOIN {T_KPI_EDITOR}  ed ON b.user_guid = ed.user_guid
    LEFT JOIN {T_KPI_EXPORT}  ex ON b.user_guid = ex.user_guid
    LEFT JOIN {T_KPI_RETURNS} rr ON b.user_guid = rr.user_guid
    -- Join engaged MAU table (Requirement 3)
    LEFT JOIN {T_ENGAGED_MAU} eng ON b.user_guid = eng.user_guid
    """)


# ============================================================
# 8.1) NUV segment memberships and aggregation
# ============================================================

def build_nuf_segment_memberships_and_agg(end_date: str):

    sql_agg = f"""
    CREATE OR REPLACE TABLE {T_NUV_USER_LEVEL} AS
    WITH nuf_base AS (
      SELECT DISTINCT
        user_guid,
        CASE WHEN lower(a.platform_category) LIKE '%android%' THEN 'android'
             WHEN lower(a.platform_category) IN ('mobile-web','dapp-windows','desktop-web','ios','android') THEN platform_category
             ELSE 'desktop-web' END AS platform_category,
        CASE WHEN market_area NOT IN ('US', 'UK', 'IN', 'BRZ') THEN 'ROW'
             ELSE market_area END AS market_area,
        source_name,
        offer_category,
        channel_detail,
        b.cloud_type,
        market_segment_category,
        contract_type,
        offer_type
      FROM {T_NUF} a LEFT JOIN ms_categorization b ON a.category_key = offer_category_key
      WHERE date_date BETWEEN date_sub('{end_date}', 27) AND '{end_date}'
    ),

    nuf_overall_members AS (
      SELECT DISTINCT user_guid, platform_category platform_cut, market_area, NULL AS segment
      FROM nuf_base
    ),

    nuf_core_members AS (
      SELECT DISTINCT user_guid, platform_category platform_cut, market_area, 'Core' AS segment
      FROM nuf_base
      WHERE {CORE_FILTER_SQL}
    ),

    nuf_edu_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'edu' AS segment
      FROM nuf_base b
      JOIN ccex_tmp.edu_mau_on_date_tmp edu
        ON b.user_guid = edu.user_guid
       AND edu.as_of_date = '{end_date}'
    ),

    nuf_3p_members AS (
      SELECT DISTINCT user_guid, b.platform_category platform_cut, market_area, '3p' AS segment
      FROM nuf_base b LEFT JOIN event_activity_on_date_vw e ON b.user_guid = e.event_user_guid
      WHERE {third_party_filter_sql}
        OR airtel_offer_id=1
    ),

    nuf_cc_entitled_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'cc entitled' AS segment
      FROM nuf_base b
      LEFT JOIN spark.cloud_type_mapping m ON b.cloud_type = m.cloud_type
      WHERE {CC_FILTER_SQL}
    ),

    nuf_photos_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'Acrobat photos' AS segment
      FROM nuf_base b
      JOIN event_activity_on_date_vw e ON b.user_guid = e.event_user_guid
      WHERE e.is_adobe_express_photos_user=true
    ),

    nuf_non_photos_members AS (
      SELECT DISTINCT b.user_guid, b.platform_category platform_cut, b.market_area, 'Acrobat non-photos' AS segment
      FROM nuf_base b
      JOIN event_activity_on_date_vw e ON b.user_guid = e.event_user_guid
      WHERE (is_acrobat_extension_user = true
             OR is_generative_reuse_user = true
             OR is_acrobat_mobile_edit_image_user = true
             OR is_enhanced_pdf_editing_user = true
             OR is_acrobat_original_creation_user = true)
    ),

    nuf_memberships AS (
      SELECT * FROM nuf_overall_members
      UNION ALL SELECT * FROM nuf_core_members
      UNION ALL SELECT * FROM nuf_edu_members
      UNION ALL SELECT * FROM nuf_3p_members
      UNION ALL SELECT * FROM nuf_cc_entitled_members
      UNION ALL SELECT * FROM nuf_non_photos_members
      UNION ALL SELECT * FROM nuf_photos_members
    ),

    nuv_agg AS (
      SELECT
        CASE WHEN GROUPING(segment)=1      THEN 'OVERALL' ELSE segment      END AS segment,
        CASE WHEN GROUPING(platform_cut)=1 THEN 'OVERALL' ELSE platform_cut END AS platform_cut,
        CASE WHEN GROUPING(market_area)=1  THEN 'OVERALL' ELSE market_area  END AS market_area,
        COUNT(DISTINCT user_guid) AS nuv_users
      FROM nuf_memberships
      GROUP BY GROUPING SETS (
        (),
        (segment),
        (platform_cut),
        (market_area),
        (segment, platform_cut),
        (segment, market_area),
        (platform_cut, market_area),
        (segment, platform_cut, market_area)
      )
    )

    SELECT * FROM nuv_agg;
    """
    spark.sql(sql_agg)


# ============================================================
# 9) Write metrics
# ============================================================

def write_metrics(end_date: str):
    seg_sql_parts = []
    for s in SEGMENTS:
        seg_sql_parts.append(f"""
        SELECT '{s["name"]}' AS segment, *
        FROM {T_USER_LEVEL}
        WHERE {s["where"]}
        """)
    seg_union = "\nUNION ALL\n".join(seg_sql_parts)

    exec_sql(f"""
    INSERT OVERWRITE TABLE {T_OUT_METRICS}
    PARTITION (end_date = '{end_date}')

    WITH seg AS (
      {seg_union}
    ),

    mau_metrics AS (
      SELECT
        CASE WHEN GROUPING(platform_cut)=1  THEN 'OVERALL' ELSE platform_cut END AS platform_cut,
        CASE WHEN GROUPING(new_or_return)=1 THEN 'OVERALL' ELSE new_or_return END AS new_or_return,
        CASE WHEN GROUPING(market_area)=1   THEN 'OVERALL' ELSE market_area END   AS market_area,
        CASE WHEN GROUPING(auth_status)=1   THEN 'OVERALL' ELSE auth_status END   AS auth_status,
        segment,

        COUNT(DISTINCT user_guid) AS MAU,
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END) AS pnuv_users,
        null total_visitors,
        CAST(NULL AS BIGINT) AS nuv_users,

        -- FTA
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_auth_flag=1 THEN user_guid END) AS d1_fta_users,
        ROUND(COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_auth_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100, 2) AS d1_fta_pct,

        COUNT(DISTINCT CASE WHEN mau_auth_flag=1 THEN user_guid END) AS fta_users,
        ROUND(COUNT(DISTINCT CASE WHEN mau_auth_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100, 2) AS fta_pct,

        -- Editor
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_editor_open=1 THEN user_guid END) AS d1_editor_users,
        ROUND(COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_editor_open=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100, 2) AS d1_editor_pct,

        COUNT(DISTINCT CASE WHEN total_editor_open=1 THEN user_guid END) AS editor_users,
        ROUND(COUNT(DISTINCT CASE WHEN total_editor_open=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100, 2) AS editor_pct,

        -- Export
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_export_flag=1 THEN user_guid END) AS d1_export_users,
        ROUND(COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d1_export_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100, 2) AS d1_export_pct,

        COUNT(DISTINCT CASE WHEN total_export_flag=1 THEN user_guid END) AS export_users,
        ROUND(COUNT(DISTINCT CASE WHEN total_export_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100, 2) AS export_pct,

        -- D2-7
        COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d2_7_return_flag=1 THEN user_guid END) AS d2_7_users,
        ROUND(COUNT(DISTINCT CASE WHEN pnuv_flag=1 AND d2_7_return_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_flag=1 THEN user_guid END), 0) * 100, 2) AS d2_7_pct,

        -- W1RR
        COUNT(DISTINCT CASE WHEN pnuv_wau=1 AND w1_return_flag=1 THEN user_guid END) AS w1_rr_users,
        ROUND(COUNT(DISTINCT CASE WHEN pnuv_wau=1 AND w1_return_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT CASE WHEN pnuv_wau=1 THEN user_guid END), 0) * 100, 2) AS w1_rr_pct,

        --- CHANGE: M1RR different for Photos vs others (Requirement 2)
        COUNT(DISTINCT CASE
            WHEN acrobat_photos_flag=1 AND pnuv_photos_flag=1 AND pnuv_photos_t4_return_flag=1 THEN user_guid
            WHEN acrobat_photos_flag=0 AND pnuv_flag=1 AND pnuv_t4_return_flag=1 THEN user_guid
            END) AS m1_rr_users,

        ROUND(
          COUNT(DISTINCT CASE
              WHEN acrobat_photos_flag=1 AND pnuv_photos_flag=1 AND pnuv_photos_t4_return_flag=1 THEN user_guid
              WHEN acrobat_photos_flag=0 AND pnuv_flag=1 AND pnuv_t4_return_flag=1 THEN user_guid
              END)
          / NULLIF(
              COUNT(DISTINCT CASE
                  WHEN acrobat_photos_flag=1 AND pnuv_photos_flag=1 THEN user_guid
                  WHEN acrobat_photos_flag=0 AND pnuv_flag=1 THEN user_guid
                  END), 0) * 100
        , 2) AS m1_rr_pct,

        -- rMAU
        COUNT(DISTINCT CASE WHEN mau_t4_return_flag=1 THEN user_guid END) AS rmau_users,
        ROUND(COUNT(DISTINCT CASE WHEN mau_t4_return_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100, 2) AS rmau_pct,

        -- Engaged MAU (Requirement 3)
        COUNT(DISTINCT CASE WHEN engaged_mau_flag=1 THEN user_guid END) AS engaged_mau,
        ROUND(COUNT(DISTINCT CASE WHEN engaged_mau_flag=1 THEN user_guid END)
          / NULLIF(COUNT(DISTINCT user_guid), 0) * 100, 2) AS engaged_mau_pct

      FROM seg
      GROUP BY segment,
        GROUPING SETS (
          (),
          (platform_cut),
          (new_or_return),
          (market_area),
          (auth_status)
        )
    )

    SELECT
      m.platform_cut,
      m.new_or_return,
      m.market_area,
      m.auth_status,
      m.segment,
      m.MAU,
      m.pnuv_users,

      CASE
        WHEN m.new_or_return='OVERALL' AND m.auth_status='OVERALL'
        THEN n.nuv_users
        ELSE NULL
      END AS nuv_users,

      total_visitors,
      m.d1_fta_users,
      m.d1_fta_pct,
      m.fta_users,
      m.fta_pct,
      m.d1_editor_users,
      m.d1_editor_pct,
      m.editor_users,
      m.editor_pct,
      m.d1_export_users,
      m.d1_export_pct,
      m.export_users,
      m.export_pct,
      m.d2_7_users,
      m.d2_7_pct,
      m.w1_rr_users,
      m.w1_rr_pct,
      m.m1_rr_users,
      m.m1_rr_pct,
      m.rmau_users,
      m.rmau_pct,
      -- Engaged MAU (Requirement 3)
      m.engaged_mau,
      m.engaged_mau_pct

    FROM mau_metrics m
    LEFT JOIN {T_NUV_USER_LEVEL} n
      ON m.platform_cut = n.platform_cut
      AND m.segment = n.segment
      AND m.market_area = n.market_area
    """)


# ============================================================
# 10) Cleanup
# ============================================================

def drop_all_temps():
    for t in [T_BASE, T_FIRST_TS, T_ACTIVITY, T_EXPORT_ACT,
              T_KPI_EDITOR, T_KPI_EXPORT, T_KPI_RETURNS,
              T_USER_LEVEL, T_ENGAGED_MAU]:
        exec_sql(f"DROP TABLE IF EXISTS {t}")

In [0]:
build_base_dedup('2026-02-27')

In [0]:
# Run full pipeline
build_first_ts('2026-02-27')
build_activity('2026-02-27')
build_export_activity('2026-02-27')
build_kpi_editor('2026-02-27')
build_kpi_export('2026-02-27')
build_kpi_returns('2026-02-27')
build_engaged_mau('2026-02-27')
build_user_level('2026-02-27')
build_event_activity_view('2026-02-27')
build_nuf_segment_memberships_and_agg('2026-02-27')
write_metrics('2026-02-27')

In [0]:

build_first_ts('2026-03-27')
build_activity('2026-03-27')
build_export_activity('2026-03-27')
build_kpi_editor('2026-03-27')
build_kpi_export('2026-03-27')
build_kpi_returns('2026-03-27')
build_engaged_mau('2026-03-27')
build_user_level('2026-03-27')
build_event_activity_view('2026-03-27')
build_nuf_segment_memberships_and_agg('2026-03-27')
write_metrics('2026-03-27')

In [0]:
%sql
SELECT
  o.segment,
  o.MAU AS old_mau,
  n.MAU AS new_mau,
  ROUND((n.MAU - o.MAU) / o.MAU * 100, 2) AS mau_diff_pct,
  o.pnuv_users AS old_pnuv,
  n.pnuv_users AS new_pnuv,
  ROUND((n.pnuv_users - o.pnuv_users) / o.pnuv_users * 100, 2) AS pnuv_diff_pct,
  o.d1_editor_pct AS old_d1_editor,
  n.d1_editor_pct AS new_d1_editor,
  o.d1_export_pct AS old_d1_export,
  n.d1_export_pct AS new_d1_export,
  o.m1_rr_pct AS old_m1rr,
  n.m1_rr_pct AS new_m1rr,
  o.rmau_pct AS old_rmau,
  n.rmau_pct AS new_rmau,
  n.engaged_mau AS engaged_mau,
  n.engaged_mau_pct AS engaged_mau_pct
FROM ccex_tmp.core_leading_indicator_metrics_mau o
JOIN ccex_tmp.core_leading_indicator_metrics_mau_v2 n
  ON o.segment = n.segment
  AND o.end_date = n.end_date
  AND o.new_or_return = n.new_or_return
  AND o.market_area = n.market_area
  AND o.auth_status = n.auth_status
  AND o.platform_cut = n.platform_cut
WHERE o.end_date = '2026-02-27'
AND o.new_or_return = 'OVERALL'
AND o.market_area = 'OVERALL'
AND o.auth_status = 'OVERALL'
AND o.platform_cut = 'OVERALL'
ORDER BY segment;

segment,old_mau,new_mau,mau_diff_pct,old_pnuv,new_pnuv,pnuv_diff_pct,old_d1_editor,new_d1_editor,old_d1_export,new_d1_export,old_m1rr,new_m1rr,old_rmau,new_rmau,engaged_mau,engaged_mau_pct
3p,1628695,1827998,12.24,1076078,1110563,3.2,71.21,70.84,17.11,17.13,0.0,31.34,0.0,32.97,1190080,65.1
Acrobat non-photos,16043883,16064138,0.13,12136679,12150366,0.11,96.52,96.51,18.57,18.58,0.0,13.84,0.0,18.95,6967362,43.37
Acrobat photos,3785152,3816324,0.82,3315923,3336262,0.61,9.95,10.2,16.3,16.36,0.0,34.95,0.0,45.32,541733,14.2
Core,19073970,19073970,0.0,14274375,14274375,0.0,56.8,56.8,13.68,13.68,0.0,12.11,0.0,19.38,9225214,48.37
OVERALL,40996782,41139485,0.35,30673834,30673834,0.0,67.52,67.52,16.31,16.31,0.0,14.2,0.0,21.91,18088582,43.97
cc entitled,2066678,2066678,0.0,363389,363389,0.0,72.4,72.4,21.71,21.71,0.0,22.89,0.0,43.15,1322075,63.97
edu,3532684,3532684,0.0,1381608,1381608,0.0,50.25,50.25,31.88,31.88,0.0,17.55,0.0,31.54,2275797,64.42


In [0]:
%sql
SELECT
  segment,
  MAU,
  pnuv_users,
  nuv_users,
  d1_editor_pct,
  d1_export_pct,
  m1_rr_pct,
  rmau_pct,
  engaged_mau,
  engaged_mau_pct
FROM ccex_tmp.core_leading_indicator_metrics_mau_v2
WHERE end_date = '2026-02-27'
AND new_or_return = 'OVERALL'
AND market_area = 'OVERALL'
AND auth_status = 'OVERALL'
AND platform_cut = 'OVERALL'
ORDER BY segment;

segment,MAU,pnuv_users,nuv_users,d1_editor_pct,d1_export_pct,m1_rr_pct,rmau_pct,engaged_mau,engaged_mau_pct
3p,1827998,1110563,1091279,70.84,17.13,31.34,32.97,1190080,65.1
Acrobat non-photos,16064138,12150366,11857501,96.51,18.58,13.84,18.95,6967362,43.37
Acrobat photos,3816324,3336262,3251074,10.2,16.36,34.95,45.32,541733,14.2
Core,19073970,14274375,25137944,56.8,13.68,12.11,19.38,9225214,48.37
OVERALL,41139485,30673834,42089918,67.52,16.31,14.2,21.91,18088582,43.97
cc entitled,2066678,363389,375557,72.4,21.71,22.89,43.15,1322075,63.97
edu,3532684,1381608,1402761,50.25,31.88,17.55,31.54,2275797,64.42


In [0]:
%sql
select * from ccex_tmp.temp_okr_cube_Data_v3 limit 20;

okr_segment,okr_subsegment,layer,segment_tag,value,fiscal_yr,fiscal_yr_and_qtr_desc,fiscal_wk_in_yr,fiscal_wk_in_qtr,fiscal_wk_starting_date,fiscal_wk_ending_date,kpi,platform_category,market_area,offer_category,entry_source,source_name,new_or_return,grp,insert_ts,date_date,key,numerator,denominator,m1rr,fiscal_year,fiscal_qtr,fiscal_week_in_qtr,fiscal_year_and_qtr_desc,pqe_fiscal_yr,pqe_fiscal_qtr,value_pqe,numerator_pqe,denominator_pqe,m1rr_pqe,value_pw,numerator_pw,denominator_pw,m1rr_pw,target,qrf,outlook,year_target
Standalone,null,null,Standalone,7,2026,2026-Q1,10,10,2026-01-31,2026-02-06,DDOM Total MAU GA,mobile-web,MEX,Trial,A.com/express-Non Image - QA,OVERALL,OVERALL,0110000,2026-04-13T01:37:15.253Z,2026-02-06,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,10,2026-Q1,2025,4,6,null,null,null,1,null,null,null,null,null,null,null
Standalone,null,null,Standalone,40954,2026,2026-Q1,03,03,2025-12-13,2025-12-19,DDOM Total MAU GA,desktop-web,OVERALL,OVERALL,First Party - CCH,OVERALL,NEW,0010110,2026-04-13T08:16:19.394Z,2025-12-19,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,3,2026-Q1,2025,4,43818,null,null,null,43644,null,null,null,null,null,null,null
Standalone,null,null,Standalone,68,2026,2026-Q1,08,08,2026-01-17,2026-01-23,DDOM Total MAU GA,desktop-web,ITA,Enterprise VIP,A.com Non Express,OVERALL,OVERALL,0110000,2026-04-13T04:04:18.991Z,2026-01-23,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,8,2026-Q1,2025,4,87,null,null,null,64,null,null,null,null,null,null,null
Standalone,null,null,Standalone,45,2026,2026-Q1,04,04,2025-12-20,2025-12-26,DDOM Total MAU GA,mobile-web,OVERALL,Individual,A.com/express-Template/Blog/Discover/Learn,OVERALL,NEW,0010010,2026-04-13T07:22:44.548Z,2025-12-26,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,4,2026-Q1,2025,4,44,null,null,null,42,null,null,null,null,null,null,null
Standalone,null,null,Standalone,5,2026,2026-Q1,05,05,2025-12-27,2026-01-02,DDOM Total MAU GA,desktop-web,NORD,Enterprise VIP,Third Party - 3P,ESDK,RETURN,0000000,2026-04-13T06:36:42.636Z,2026-01-02,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,5,2026-Q1,2025,4,16,null,null,null,8,null,null,null,null,null,null,null
Standalone,null,null,Standalone,2,2026,2026-Q1,04,04,2025-12-20,2025-12-26,DDOM Total MAU GA,mobile-web,GER,OVERALL,A.com/express-Create,ESDK,NUV,0000100,2026-04-13T07:22:44.548Z,2025-12-26,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,4,2026-Q1,2025,4,null,null,null,null,2,null,null,null,null,null,null,null
Standalone,null,null,Standalone,14,2026,2026-Q1,10,10,2026-01-31,2026-02-06,DDOM Total MAU GA,OVERALL,NORD,OVERALL,Third Party - PWA,OVERALL,NUV,0010101,2026-04-13T01:37:15.253Z,2026-02-06,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,10,2026-Q1,2025,4,45,null,null,null,19,null,null,null,null,null,null,null
Standalone,null,null,Standalone,5139,2026,2026-Q1,02,02,2025-12-06,2025-12-12,DDOM Total MAU GA,desktop-web,NORD,Edu Enterprise K12,D2P,CCEX,RETURN,0000000,2026-04-13T09:11:21.837Z,2025-12-12,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,2,2026-Q1,2025,4,4946,null,null,null,5059,null,null,null,null,null,null,null
Standalone,null,null,Standalone,408,2026,2026-Q1,09,09,2026-01-24,2026-01-30,DDOM Total MAU GA,android,EE,Trial,Mobile,CCEX,RETURN,0000000,2026-04-13T02:53:25.209Z,2026-01-30,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,9,2026-Q1,2025,4,389,null,null,null,406,null,null,null,null,null,null,null
Standalone,null,null,Standalone,15,2026,2026-Q1,04,04,2025-12-20,2025-12-26,DDOM Total MAU GA,OVERALL,BRZ,CC Free,A.com/express-Image - QA,ESDK,RETURN,0000001,2026-04-13T07:22:44.548Z,2025-12-26,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,4,2026-Q1,2025,4,17,null,null,null,16,null,null,null,null,null,null,null


In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "400")

In [0]:
%sql
-- Count with 28-day window preconsent filter
SELECT COUNT(DISTINCT m.user_guid)
FROM ccex_tmp.okr_segment_tags_final_base m
LEFT ANTI JOIN ( select * from ccex.ccex_tron_preconsent_deduped_mau where date_date='2026-02-27') pc
  ON m.user_guid = pc.visitor_guid
  --AND pc.date_date =m.date_date
  and lower(m.segment_tag) in ("non-consent","standalone")

-- LEFT anti JOIN ccex_tmp.third_party_nonconsent_mau_on_date_tmp nc
--   ON m.user_guid = nc.user_guid
--   AND nc.as_of_date = '2026-02-27'

WHERE m.date_date = '2026-02-27'
-- and segment_tag not in ('non-consent')
-- AND m.platform_category IS NOT NULL
-- AND m.new_or_return IS NOT NULL
-- AND pc.visitor_guid IS NULL
-- AND nc.user_guid IS NULL;

COUNT(DISTINCTm.user_guid)
41139485


In [0]:
%sql
select * from ccex_tmp.temp_okr_cube_Data_v3 limit 10;

okr_segment,okr_subsegment,layer,segment_tag,value,fiscal_yr,fiscal_yr_and_qtr_desc,fiscal_wk_in_yr,fiscal_wk_in_qtr,fiscal_wk_starting_date,fiscal_wk_ending_date,kpi,platform_category,market_area,offer_category,entry_source,source_name,new_or_return,grp,insert_ts,date_date,key,numerator,denominator,m1rr,fiscal_year,fiscal_qtr,fiscal_week_in_qtr,fiscal_year_and_qtr_desc,pqe_fiscal_yr,pqe_fiscal_qtr,value_pqe,numerator_pqe,denominator_pqe,m1rr_pqe,value_pw,numerator_pw,denominator_pw,m1rr_pw,target,qrf,outlook,year_target
Acrobat - Non Photos,null,null,Acrobat - Non Photos,692,2026,2026-Q2,23,10,2026-05-02,2026-05-08,DDOM Total MAU GA,desktop-web,BEN,Edu Enterprise HED,Acrobat - Others,CCEX,RETURN,0000000,2026-05-13T22:35:12.150Z,2026-05-08,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,10,2026-Q2,2026,1,545,null,null,null,673,null,null,null,null,null,null,null
Overall,null,null,OVERALL,11,2026,2026-Q2,14,01,2026-02-28,2026-03-06,DDOM Total MAU GA,mobile-web,MED,CC Free,D2P,OVERALL,NEW,1010000,2026-04-12T20:08:59.711Z,2026-03-06,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,1,2026-Q2,2026,1,8,null,null,null,8,null,null,null,null,null,null,null
Express - Photos,null,null,Express - Photos,904,2026,2026-Q2,14,01,2026-02-28,2026-03-06,DDOM Total MAU GA,desktop-web,IBE,OVERALL,D2P,CCEX,OVERALL,0100100,2026-04-12T20:08:59.711Z,2026-03-06,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,1,2026-Q2,2026,1,890,null,null,null,890,null,null,null,null,null,null,null
Overall,null,null,OVERALL,2323,2026,2026-Q2,17,04,2026-03-21,2026-03-27,DDOM Total MAU GA,OVERALL,ME,Trial,OVERALL,OVERALL,RETURN,1011001,2026-04-17T13:32:15.957Z,2026-03-27,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,4,2026-Q2,2026,1,4934,null,null,null,2613,null,null,null,null,null,null,null
Acrobat - Non Photos,null,null,Acrobat - Non Photos,1,2026,2026-Q2,19,06,2026-04-04,2026-04-10,DDOM Total MAU GA,desktop-web,GER,Unauthenticated,First Party - Other,OVERALL,NUV,0010000,2026-04-17T11:41:39.104Z,2026-04-10,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,6,2026-Q2,2026,1,null,null,null,null,1,null,null,null,null,null,null,null
Standalone,null,null,Standalone,28,2026,2026-Q2,18,05,2026-03-28,2026-04-03,DDOM Total MAU GA,desktop-web,UNKNOWN,Edu Enterprise K12,Other,OVERALL,OVERALL,0110000,2026-04-17T12:42:33.904Z,2026-04-03,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,5,2026-Q2,2026,1,29,null,null,null,30,null,null,null,null,null,null,null
Standalone,null,null,Standalone,1,2026,2026-Q1,03,03,2025-12-13,2025-12-19,DDOM Total MAU GA,desktop-web,CAN,UNKNOWN,First Party - CCH,CCEX,RETURN,0000000,2026-04-13T08:16:19.394Z,2025-12-19,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,3,2026-Q1,2025,4,1,null,null,null,1,null,null,null,null,null,null,null
Acrobat - Non Photos,null,null,Acrobat - Non Photos,6,2026,2026-Q2,20,07,2026-04-11,2026-04-17,DDOM Total MAU GA,desktop-web,CHN,Team,D2P,CCEX,NEW,0000000,2026-04-21T06:33:55.647Z,2026-04-17,DDOM_TOTAL_MAU_GA,null,null,null,2026,2,7,2026-Q2,2026,1,2,null,null,null,6,null,null,null,null,null,null,null
Overall,null,null,OVERALL,18396,2026,2026-Q1,04,04,2025-12-20,2025-12-26,DDOM Total MAU GA,desktop-web,SLAM,OVERALL,D2P,CCEX,NEW,1000100,2026-04-13T07:22:44.548Z,2025-12-26,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,4,2026-Q1,2025,4,22511,null,null,null,20901,null,null,null,null,null,null,null
Overall,null,null,OVERALL,2,2026,2026-Q1,13,13,2026-02-21,2026-02-27,DDOM Total MAU GA,android,FRA,CC Free,A.com/express-Create,CCEX,RETURN,1000000,2026-04-12T21:24:08.418Z,2026-02-27,DDOM_TOTAL_MAU_GA,null,null,null,2026,1,13,2026-Q1,2025,4,null,null,null,null,3,null,null,null,null,null,null,null
